## Phần 2


In [1]:
import spacy
from spacy import displacy

In [2]:
nlp = spacy.load("en_core_web_md")

In [3]:
text = "The quick brown fox jumps over the lazy dog."

In [4]:
doc = nlp(text)

In [5]:
displacy.serve(doc, style="dep")

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\spacy\displacy\__init__.py:108: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'dep' visualizer
Serving on http://0.0.0.0:5000 ...

Shutting down server on port 5000.




**1. Từ nào là gốc (ROOT) của câu?**
- **jumps** là ROOT của câu. Đây là động từ chính của câu, thể hiện hành động chính.

**2. jumps có những từ phụ thuộc (dependent) nào? Các quan hệ đó là gì?**
- **fox** - quan hệ `nsubj` (nominal subject): chủ ngữ của động từ
- **over** - quan hệ `prep` (preposition): giới từ bổ nghĩa cho động từ

**3. fox là head của những từ nào?**
- **The** - quan hệ `det` (determiner): mạo từ xác định
- **quick** - quan hệ `amod` (adjectival modifier): tính từ bổ nghĩa
- **brown** - quan hệ `amod` (adjectival modifier): tính từ bổ nghĩa

## Phần 3 


In [6]:
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)

print(f"{'TEXT':<12} | {'DEP':<10} | {'HEAD TEXT':<12} | {'HEAD POS':<8} | {'CHILDREN'}")
print("-" * 70)

TEXT         | DEP        | HEAD TEXT    | HEAD POS | CHILDREN
----------------------------------------------------------------------


In [7]:
for token in doc:
# Trích xuất các thuộc tính
    children = [child.text for child in token.children]
    print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12} | {token.head.pos_:<8} | {children}")

Apple        | nsubj      | looking      | VERB     | []
is           | aux        | looking      | VERB     | []
looking      | ROOT       | looking      | VERB     | ['Apple', 'is', 'at']
at           | prep       | looking      | VERB     | ['buying']
buying       | pcomp      | at           | ADP      | ['startup']
U.K.         | compound   | startup      | NOUN     | []
startup      | dobj       | buying       | VERB     | ['U.K.', 'for']
for          | prep       | startup      | NOUN     | ['billion']
$            | quantmod   | billion      | NUM      | []
1            | compound   | billion      | NUM      | []
billion      | pobj       | for          | ADP      | ['$', '1']


# Phan 4


In [8]:
text = "The cat chased the mouse and the dog watched them."
doc = nlp(text)

for token in doc:
    if token.pos_ == "VERB":
        verb = token.text
        subjects = ""
        obj = ""
        for child in token.children:
            if child.dep_ == "nsubj":
                subject = child.text
            if child.dep_ == "dobj":
                obj = child.text
        if subject and obj:
            print(f"Found Triplet: ({subject}, {verb}, {obj})")

Found Triplet: (cat, chased, mouse)
Found Triplet: (dog, watched, them)


In [9]:
text = "The big, fluffy white cat is sleeping on the warm mat."
doc = nlp(text)
for token in doc:
    # Chỉ tìm các danh từ
    if token.pos_ == "NOUN":
        adjectives = []
        # Tìm các tính từ bổ nghĩa (amod) trong các con của danh từ
        for child in token.children:
            if child.dep_ == "amod":
                adjectives.append(child.text)
        if adjectives:
            print(f"Danh từ '{token.text}' được bổ nghĩa bởi các tính từ: {adjectives}")

Danh từ 'cat' được bổ nghĩa bởi các tính từ: ['big', 'fluffy', 'white']
Danh từ 'mat' được bổ nghĩa bởi các tính từ: ['warm']


## Phần 5

In [13]:
# Bài 1: Tìm động từ chính của câu
def find_main_verb(doc):
    """
    Tìm động từ chính của câu (token có quan hệ ROOT)
    
    Args:
        doc: đối tượng Doc của spaCy
    Returns:
        Token là động từ chính hoặc None nếu không tìm thấy
    """
    for token in doc:
        if token.dep_ == "ROOT":
            return token
    return None

# Test với các câu khác nhau
test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Apple is looking at buying U.K. startup for $1 billion",
    "The cat chased the mouse and the dog watched them.",
    "She has been studying for the exam all day."
]


for sentence in test_sentences:
    doc = nlp(sentence)
    main_verb = find_main_verb(doc)
    print(f"\nCâu: '{sentence}'")
    if main_verb:
        print(f"  → Động từ chính (ROOT): '{main_verb.text}' (POS: {main_verb.pos_})")
    else:
        print("  → Không tìm thấy động từ chính")


Câu: 'The quick brown fox jumps over the lazy dog.'
  → Động từ chính (ROOT): 'jumps' (POS: VERB)

Câu: 'Apple is looking at buying U.K. startup for $1 billion'
  → Động từ chính (ROOT): 'looking' (POS: VERB)

Câu: 'The cat chased the mouse and the dog watched them.'
  → Động từ chính (ROOT): 'chased' (POS: VERB)

Câu: 'She has been studying for the exam all day.'
  → Động từ chính (ROOT): 'studying' (POS: VERB)


In [14]:
# Bài 2: Trích xuất các cụm danh từ (Noun Chunks)
def get_noun_chunks(doc):
    """
    Trích xuất các cụm danh từ từ câu.
    Một cụm danh từ gồm danh từ và các từ bổ nghĩa cho nó (det, amod, compound, poss, nummod)
    
    Args:
        doc: đối tượng Doc của spaCy
    Returns:
        List các cụm danh từ (mỗi cụm là một string)
    """
    noun_chunks = []
    
    # Tìm tất cả các danh từ (NOUN, PROPN)
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            # Thu thập tất cả các từ thuộc cụm danh từ này
            chunk_tokens = []
            
            # Duyệt tất cả token trong doc để tìm các từ bổ nghĩa
            for t in doc:
                # Kiểm tra xem t có phải là bổ nghĩa trực tiếp cho danh từ này không
                if t.head == token and t.dep_ in ["det", "amod", "compound", "poss", "nummod", "punct"]:
                    chunk_tokens.append(t)
                # Hoặc chính danh từ đó
                elif t == token:
                    chunk_tokens.append(t)
            
            # Sắp xếp theo thứ tự xuất hiện trong câu
            chunk_tokens.sort(key=lambda x: x.i)
            
            # Tạo chuỗi cụm danh từ
            if chunk_tokens:
                chunk_text = " ".join([t.text for t in chunk_tokens])
                # Tránh trùng lặp
                if chunk_text not in noun_chunks:
                    noun_chunks.append(chunk_text)
    
    return noun_chunks


def get_noun_chunks_v2(doc):
    """
    Phiên bản 2: Duyệt từ danh từ và thu thập tất cả subtree bên trái
    """
    noun_chunks = []
    seen_heads = set()
    
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"] and token not in seen_heads:
            seen_heads.add(token)
            
            # Thu thập tất cả children có dep phù hợp + chính token đó
            chunk_tokens = [token]
            
            for child in token.children:
                if child.dep_ in ["det", "amod", "compound", "poss", "nummod", "nmod"]:
                    chunk_tokens.append(child)
                    # Cũng lấy các compound của compound (nested)
                    for grandchild in child.children:
                        if grandchild.dep_ in ["compound", "amod"]:
                            chunk_tokens.append(grandchild)
            
            # Sắp xếp theo vị trí
            chunk_tokens.sort(key=lambda x: x.i)
            chunk_text = " ".join([t.text for t in chunk_tokens])
            noun_chunks.append(chunk_text)
    
    return noun_chunks


# Test
test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "The big, fluffy white cat is sleeping on the warm mat.",
    "Apple Inc. announced a new iPhone model yesterday."
]


for sentence in test_sentences:
    doc = nlp(sentence)
    
    print(f"\nCâu: '{sentence}'")
    
    # So sánh với noun_chunks có sẵn của spaCy
    spacy_chunks = [chunk.text for chunk in doc.noun_chunks]
    print(f"  spaCy noun_chunks: {spacy_chunks}")
    
    # Kết quả từ hàm tự viết
    my_chunks = get_noun_chunks(doc)
    print(f"  Hàm tự viết v1:    {my_chunks}")
    
    my_chunks_v2 = get_noun_chunks_v2(doc)
    print(f"  Hàm tự viết v2:    {my_chunks_v2}")


Câu: 'The quick brown fox jumps over the lazy dog.'
  spaCy noun_chunks: ['The quick brown fox', 'the lazy dog']
  Hàm tự viết v1:    ['The quick brown fox', 'the lazy dog']
  Hàm tự viết v2:    ['The quick brown fox', 'the lazy dog']

Câu: 'The big, fluffy white cat is sleeping on the warm mat.'
  spaCy noun_chunks: ['The big, fluffy white cat', 'the warm mat']
  Hàm tự viết v1:    ['The big , fluffy white cat', 'the warm mat']
  Hàm tự viết v2:    ['The big fluffy white cat', 'the warm mat']

Câu: 'Apple Inc. announced a new iPhone model yesterday.'
  spaCy noun_chunks: ['Apple Inc.', 'a new iPhone model']
  Hàm tự viết v1:    ['Apple', 'Apple Inc.', 'iPhone', 'a new iPhone model', 'yesterday']
  Hàm tự viết v2:    ['Apple', 'Apple Inc.', 'iPhone', 'a new iPhone model', 'yesterday']


In [15]:
# Bài 3: Tìm đường đi ngắn nhất trong cây (từ token đến ROOT)
def get_path_to_root(token):
    """
    Tìm đường đi từ một token bất kỳ lên đến gốc (ROOT) của cây.
    
    Args:
        token: Token của spaCy
    Returns:
        List các token trên đường đi (bao gồm cả token đầu vào và ROOT)
    """
    path = [token]
    current = token
    
    # Duyệt ngược lên head cho đến khi gặp ROOT (head của ROOT là chính nó)
    while current.head != current:  # ROOT có đặc điểm: token.head == token
        current = current.head
        path.append(current)
    
    return path


def get_path_between_tokens(token1, token2):
    """
    Tìm đường đi ngắn nhất giữa hai token bất kỳ trong cây dependency.
    
    Args:
        token1, token2: Hai Token của spaCy
    Returns:
        List các token trên đường đi
    """
    # Lấy đường đi từ mỗi token đến ROOT
    path1 = get_path_to_root(token1)
    path2 = get_path_to_root(token2)
    
    # Tìm điểm giao nhau (Lowest Common Ancestor - LCA)
    path1_set = set(path1)
    lca = None
    lca_idx_in_path2 = -1
    
    for i, t in enumerate(path2):
        if t in path1_set:
            lca = t
            lca_idx_in_path2 = i
            break
    
    if lca is None:
        return []  # Không tìm thấy đường đi (lỗi)
    
    # Xây dựng đường đi: token1 -> ... -> LCA -> ... -> token2
    lca_idx_in_path1 = path1.index(lca)
    
    # Đường từ token1 đến LCA (không bao gồm LCA để tránh trùng)
    path_to_lca = path1[:lca_idx_in_path1]
    
    # Đường từ LCA đến token2 (đảo ngược)
    path_from_lca = path2[:lca_idx_in_path2 + 1][::-1]
    
    return path_to_lca + path_from_lca


# Test
test_sentence = "The quick brown fox jumps over the lazy dog."
doc = nlp(test_sentence)


print(f"\nCâu: '{test_sentence}'")
print("\n--- Đường đi từ mỗi token đến ROOT ---")

for token in doc:
    path = get_path_to_root(token)
    path_str = " → ".join([f"{t.text}({t.dep_})" for t in path])
    print(f"  '{token.text}': {path_str}")

# Test đường đi giữa hai token
print("\n--- Đường đi giữa hai token bất kỳ ---")
token_pairs = [
    ("The", "dog"),      # từ đầu đến cuối
    ("quick", "lazy"),   # hai tính từ
    ("fox", "jumps"),    # chủ ngữ đến động từ
]

for t1_text, t2_text in token_pairs:
    token1 = [t for t in doc if t.text == t1_text][0]
    token2 = [t for t in doc if t.text == t2_text][0]
    
    path = get_path_between_tokens(token1, token2)
    path_str = " → ".join([t.text for t in path])
    print(f"  '{t1_text}' ↔ '{t2_text}': {path_str}")


Câu: 'The quick brown fox jumps over the lazy dog.'

--- Đường đi từ mỗi token đến ROOT ---
  'The': The(det) → fox(nsubj) → jumps(ROOT)
  'quick': quick(amod) → fox(nsubj) → jumps(ROOT)
  'brown': brown(amod) → fox(nsubj) → jumps(ROOT)
  'fox': fox(nsubj) → jumps(ROOT)
  'jumps': jumps(ROOT)
  'over': over(prep) → jumps(ROOT)
  'the': the(det) → dog(pobj) → over(prep) → jumps(ROOT)
  'lazy': lazy(amod) → dog(pobj) → over(prep) → jumps(ROOT)
  'dog': dog(pobj) → over(prep) → jumps(ROOT)
  '.': .(punct) → jumps(ROOT)

--- Đường đi giữa hai token bất kỳ ---
  'The' ↔ 'dog': The → fox → jumps → over → dog
  'quick' ↔ 'lazy': quick → fox → jumps → over → dog → lazy
  'fox' ↔ 'jumps': fox → jumps
